## ITSS 4382.002 - Group 1
## Project 2 – Market Basket Analysis for FreshMart


In [ ]:
from google.colab import files
uploaded = files.upload()


In [ ]:
import pandas as pd

# Load the 'FreshMart_SalesData.csv' file into a DataFrame named `df`
df = pd.read_csv('/content/FreshMart_SalesData.csv')

# Display the first 5 rows of the DataFrame
print("First 5 rows of the DataFrame:")
print(df.head())

# Print a concise summary of the DataFrame
print("\nDataFrame Info:")
df.info()

# Generate descriptive statistics of the numerical columns
print("\nDescriptive Statistics:")
print(df.describe())

## Prepare Data for Market Basket Analysis


In [ ]:
transactions = df.groupby('TransactionID')['ProductCategory'].apply(list)
print("Transactions in list format (first 5):")
print(transactions.head())

## Perform Market Basket Analysis (Association Rules)




In [ ]:
import pandas as pd

!pip install mlxtend

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

# Redefine transactions: group by CustomerID to get all categories bought by each customer
transactions = df.groupby('CustomerID')['ProductCategory'].apply(list)

# Convert transactions Series to a list of lists
transactions_list = transactions.tolist()

# Initialize and fit TransactionEncoder
te = TransactionEncoder()
te_ary = te.fit(transactions_list).transform(transactions_list)
df_transactions_encoded = pd.DataFrame(te_ary, columns=te.columns_)

# Apply Apriori algorithm with a lower min_support
frequent_itemsets = apriori(df_transactions_encoded, min_support=0.001, use_colnames=True)

# Generate association rules with a lower confidence threshold
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.1)
rules = rules.sort_values(by='lift', ascending=False)

print("First 5 association rules (sorted by Lift):")
print(rules.head())

## Data for K-Means clustering



In [ ]:
df['TotalSales'] = df['Quantity'] * df['Price']
customer_features = df.groupby('CustomerID').agg(
    TotalSpending=('TotalSales', 'sum'),
    PurchaseFrequency=('TransactionID', 'nunique')
).reset_index()

print("Customer Features for K-Means Clustering (first 5 rows):")
print(customer_features.head())

Scaling the `TotalSpending` and `PurchaseFrequency` features



In [ ]:
from sklearn.preprocessing import StandardScaler

# Initialize the StandardScaler
scaler = StandardScaler()

# Scale the customer features
X = customer_features[['TotalSpending', 'PurchaseFrequency']]
X_scaled = scaler.fit_transform(X)

# Convert the scaled features back to a DataFrame for better readability
X_scaled_df = pd.DataFrame(X_scaled, columns=['TotalSpending_Scaled', 'PurchaseFrequency_Scaled'])

print("Scaled Customer Features (first 5 rows):")
print(X_scaled_df.head())


Determining the optimal number of clusters for K-Means



In [ ]:
from sklearn.cluster import KMeans

wcss = []
for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, init='k-means++', random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)


K-Means model with 4 clusters to the scaled customer features.



In [ ]:
kmeans_model = KMeans(n_clusters=4, init='k-means++', random_state=42, n_init=10)
customer_features['Cluster'] = kmeans_model.fit_predict(X_scaled)

print("Customer Features with assigned Clusters (first 5 rows):")
print(customer_features.head())

Mean 'TotalSpending' and 'PurchaseFrequency' for each cluster.



In [ ]:
cluster_summary = customer_features.groupby('Cluster').agg(
    MeanSpending=('TotalSpending', 'mean'),
    MeanPurchaseFrequency=('PurchaseFrequency', 'mean'),
    CustomerCount=('CustomerID', 'count')
).reset_index()

print("Cluster Summary:")
print(cluster_summary)

#Market Basket Analysis - Muhammed Hasan Abbasi




In [ ]:
import matplotlib.pyplot as plt

# Prepare readable labels (antecedent → consequent)
rules['rule_text'] = rules['antecedents'].astype(str) + " ➜ " + rules['consequents'].astype(str)

# Select top 10 rules by lift
top10 = rules.sort_values('lift', ascending=False).head(10)

# Plot colored bar chart
plt.figure(figsize=(14, 8))
colors = plt.cm.tab10(range(10))  # 10 different colors

plt.barh(top10['rule_text'], top10['lift'], color=colors)
plt.xlabel("Lift")
plt.title("Top 10 Recommended Bundle Offers (Based on Lift)")
plt.gca().invert_yaxis()

plt.show()




# Most Frequently Purchased Categories - Sarim Suhrwardy

In [ ]:
import pandas as pd
rules_q1 = rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].copy()
def make_pair(row):
    items = list(row['antecedents']) + list(row['consequents'])
    return tuple(sorted(items))

rules_q1['item_pair'] = rules_q1.apply(make_pair, axis=1)
pair_summary = (
    rules_q1
    .groupby('item_pair')
    .agg(
        pair_support=('support', 'max'),
        avg_confidence=('confidence', 'mean'),
        max_lift=('lift', 'max')
    )
    .reset_index()
)
top_pairs = pair_summary.sort_values('pair_support', ascending=False).head(10)
top_pairs['item_pair'] = top_pairs['item_pair'].apply(lambda x: " & ".join(x))
print("Top 10 most frequently purchased-together product/category pairs:")
print(top_pairs)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
rules_pairs = rules[['antecedents', 'consequents', 'support']].copy()
def make_pair_label(row):
    items = list(row['antecedents']) + list(row['consequents'])
    items = sorted(set(items))
    return " & ".join(items)
rules_pairs['pair'] = rules_pairs.apply(make_pair_label, axis=1)
pair_support = (
    rules_pairs
    .groupby('pair')['support']
    .max()
    .reset_index()
)
top_pairs = (
    pair_support
    .sort_values('support', ascending=False)
    .head(10)
    .reset_index(drop=True)
)
sns.set(style="whitegrid")
freshmart_colors = ["#F28C58", "#FDCFB7", "#88B788", "#F7A072", "#E38B29"]
plt.figure(figsize=(12, 7))
ax = sns.barplot(
    data=top_pairs,
    x='support',
    y='pair',
    palette=freshmart_colors
)
plt.title("Top 10 Frequently Purchased-Together Category Pairs",
          fontsize=16, color="#D96F42")
plt.xlabel("Support (Fraction of Customers / Baskets)", fontsize=12)
plt.ylabel("Category Pair", fontsize=12)
for j, row in top_pairs.iterrows():
    ax.text(
        row['support'] + 0.005,
        j,
        f"{row['support']:.3f}",
        va='center',
        fontsize=7
    )
plt.tight_layout()
plt.show()